# Exploratory Data Analysis (Part 2) - Post-Preprocessing

After running our automated preprocessing pipeline (`poetry run task preprocess`), we have generated `data/processed/clean_traffic_data.csv`.

This dataset contains our new engineered features:
- Cyclical time representations (`hour_sin`, `hour_cos`, etc.)
- Binary flags (`is_weekend`, `is_rush_hour`, `is_holiday`)
- Robust weather flags (`is_raining`, `is_snowing`, `is_foggy_misty`)

**Objective:**
Analyze the relationships between these new features, check for multicollinearity (especially between the original text categories and our new binary flags), and perform feature selection to decide which columns will be fed into the final Machine Learning models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('deep')

# Load the processed dataset
df = pd.read_csv('../data/processed/clean_traffic_data.csv', parse_dates=['date_time'], index_col='date_time')
df.head()

## 1. Feature Correlation & Multicollinearity

Let's look at the correlation matrix for our numerical and boolean features. High correlation between predictor variables indicates multicollinearity, which can harm linear models and make feature importance difficult to interpret.

In [ ]:
# Select numerical and boolean columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

plt.figure(figsize=(14, 10))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Features', fontsize=16)
plt.show()

### Initial Observations from Correlation:
1. **Cyclical Features**: We expect `hour_sin` and `hour_cos` to have interesting non-linear relationships with the target. 
2. **Multicollinearity Check**: Look closely at correlations between variables like `temp` and `month`, or `is_raining` and `rain_1h`. If they are highly correlated, we might drop one.

## 2. Analyzing New Engineered Features
Let's visually validate our engineered features to ensure they capture the variance in `traffic_volume`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df, x='is_weekend', y='traffic_volume', ax=axes[0])
axes[0].set_title('Traffic Volume: Weekday (0) vs Weekend (1)')

sns.boxplot(data=df, x='is_rush_hour', y='traffic_volume', ax=axes[1])
axes[1].set_title('Traffic Volume: Non-Rush (0) vs Rush Hour (1)')

plt.show()

The boxplots should confirm our intuition: traffic is significantly lower on weekends, and much higher and concentrated during rush hours.

## 3. Weather Analysis: Raw Categories vs. Engineered Flags

We have the original `weather_main` categorical column and our custom boolean flags (`is_raining`, `is_snowing`, `is_foggy_misty`).

Let's check if the custom flags capture the essential weather information, which might allow us to drop the high-cardinality categorical strings.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=df, x='is_raining', y='traffic_volume', ax=axes[0])
axes[0].set_title('Traffic by Rain Flag')

sns.barplot(data=df, x='is_snowing', y='traffic_volume', ax=axes[1])
axes[1].set_title('Traffic by Snow Flag')

sns.barplot(data=df, x='is_foggy_misty', y='traffic_volume', ax=axes[2])
axes[2].set_title('Traffic by Fog/Mist Flag')

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='weather_main', y='traffic_volume')
plt.title('Traffic Volume by Original weather_main Category')
plt.xticks(rotation=45)
plt.show()

## 4. Feature Selection Decisions

Based on the EDA above, we must decide which columns to keep for the model.

**Candidates for removal:**
1. `holiday`: Mostly `NaN` or "None". We created `is_holiday` which is a clean binary representation. We can drop `holiday`.
2. `weather_description`: High cardinality, lots of typos/redundancies. We captured its essence in `is_raining`, `is_snowing`, and `is_foggy_misty`. We can drop `weather_description`.
3. `weather_main`: If we decide our boolean flags are sufficient, we can drop `weather_main` entirely to avoid One-Hot Encoding and simplify the model. Alternatively, we can keep it and One-Hot Encode it, but drop the boolean flags. We will evaluate this trade-off.
4. `hour`, `day_of_week`, `month`, `year`: We have `hour_sin`, `hour_cos`, `day_sin`, `day_cos`. We should likely drop the raw integer representations to prevent the model from learning linear relationships on cyclic data.

**Next Step for the User:**
Run this notebook, interpret the graphs, and confirm the list of features to drop in the final ML preparation script.